# 팀 코드 검토 후 RG3 시점별 센서 관계 실험
hayeon 브랜치 전용. RH 가스만, 병합 없이 진행. 기존 RF 기준과 구간 홀드아웃/과거순 평가를 비교한다.

In [1]:
from pathlib import Path
import json, subprocess
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, average_precision_score, f1_score
root=Path.cwd()
if not (root/'data').exists(): root=root.parent
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip()=='hayeon'
old=json.loads((root/'notebooks/10_original_defect_report_revision.ipynb').read_text(encoding='utf-8'))
exec(''.join(next(c for c in old['cells'] if c['cell_type']=='code')['source']).split('for feature in features:')[0])
shots=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=shots.loc[shots['product'].eq('RG3') & (shots.PassOrFail.eq(0)|shots.Reason.str.contains('가스',na=False))].sort_values('TimeStamp').reset_index(drop=True)
rh=labels.loc[labels['product'].eq('RG3')&labels.side.eq('RH')]
assert len(d)==583 and d.PassOrFail.sum()==17
sidecheck=d[keys+['PassOrFail']].merge(rh[keys+['PassOrFail']],on=keys,validate='one_to_one')
assert sidecheck.PassOrFail_x.equals(sidecheck.PassOrFail_y)
segments={k:g.sort_values('TimeStamp').reset_index(drop=True) for k,g in observed.loc[observed['product'].eq('RG3')].groupby('history_segment')}
def rz(x,p):
    p=np.asarray(p,float); p=p[np.isfinite(p)]
    if len(p)<5 or not np.isfinite(x): return np.nan
    scale=max(float(np.std(p)),float(1.4826*np.median(np.abs(p-np.median(p)))))
    # Constant history has no reliable standardized departure: leave missing, do not force zero.
    if scale<1e-6: return np.nan
    return float(np.clip((x-np.median(p))/scale,-20,20))
out=[]
for _,row in d.iterrows():
    g=segments[row.history_segment]
    i=int(np.searchsorted(g.TimeStamp.values,np.datetime64(row.TimeStamp),side='left'))
    p=g.iloc[max(0,i-30):i]
    assert p.empty or p.TimeStamp.max()<row.TimeStamp
    vals={'local_pressure_relation_z':np.nan,'local_cushion_z':rz(row.Cushion_Position,p.Cushion_Position.tail(10)),
          'local_barrel2_minus5_z':rz(row.Barrel_Temperature_2-row.Barrel_Temperature_5,p.Barrel_Temperature_2-p.Barrel_Temperature_5),
          'pressure_relation_fit_n':0,'pressure_relation_cal_n':0}
    rel=p[['Max_Injection_Speed','Max_Injection_Pressure']].dropna()
    if len(rel)>=15:
        train=rel.iloc[:-5]; cal=rel.iloc[-5:]
        mx=train.Max_Injection_Speed.mean(); sx=train.Max_Injection_Speed.std(ddof=0)
        my=train.Max_Injection_Pressure.mean(); sy=train.Max_Injection_Pressure.std(ddof=0)
        if sx>1e-6 and sy>1e-6:
            m=Ridge(alpha=1.0).fit(((train.Max_Injection_Speed.values-mx)/sx).reshape(-1,1),(train.Max_Injection_Pressure.values-my)/sy)
            cal_res=(cal.Max_Injection_Pressure.values-my)/sy-m.predict(((cal.Max_Injection_Speed.values-mx)/sx).reshape(-1,1))
            now_res=(row.Max_Injection_Pressure-my)/sy-m.predict(np.array([[(row.Max_Injection_Speed-mx)/sx]]))[0]
            vals['local_pressure_relation_z']=rz(now_res,cal_res)
            vals['pressure_relation_fit_n']=len(train); vals['pressure_relation_cal_n']=5
    out.append(vals)
derived=pd.DataFrame(out)
d=pd.concat([d,derived],axis=1)
baseline=['Barrel_Temperature_2__past10_std','Hopper_Temperature__past10_std','Cushion_Position__past10_std','Barrel_Temperature_5__past10_std']
sets={
'baseline_std4':baseline,
'local_relation3':['local_pressure_relation_z','local_cushion_z','local_barrel2_minus5_z'],
'local_relation_plus_std4':['local_pressure_relation_z','local_cushion_z','Barrel_Temperature_2__past10_std','Barrel_Temperature_5__past10_std']}
# Predefined small candidate list. No threshold optimization on test labels.
d.to_csv(tables/'rg3_local_relation_features.csv',index=False,encoding='utf-8-sig')
coverage=d.groupby('PassOrFail')[list(derived.columns[:3])].count()
coverage.to_csv(tables/'rg3_local_relation_coverage.csv',encoding='utf-8-sig')
blocks=d.comparison_block.drop_duplicates().tolist()
y=d.PassOrFail.to_numpy(int)
metrics=[]; predictions=[]; foldlog=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for name,fs in sets.items():
        prob=np.full(len(d),np.nan)
        for j,b in enumerate(blocks):
            te=d.comparison_block.eq(b).to_numpy()
            tr=(~te) if scheme=='leave_one_block_out' else d.comparison_block.isin(blocks[:j]).to_numpy()
            if len(np.unique(y[tr]))<2:
                foldlog.append(dict(scheme=scheme,model=name,block=b,status='not evaluated: no prior two-class training',train=int(tr.sum()),train_gas=int(y[tr].sum()),test=int(te.sum()),test_gas=int(y[te].sum())))
                continue
            model=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=-1))
            model.fit(d.loc[tr,fs],y[tr])
            prob[te]=model.predict_proba(d.loc[te,fs])[:,1]
            foldlog.append(dict(scheme=scheme,model=name,block=b,status='evaluated',train=int(tr.sum()),train_gas=int(y[tr].sum()),test=int(te.sum()),test_gas=int(y[te].sum())))
        ok=np.isfinite(prob)
        yy=y[ok]; pp=prob[ok]; pred=pp>=.5
        tn,fp,fn,tp=confusion_matrix(yy,pred,labels=[0,1]).ravel()
        inspections=0; caught=0
        for b in blocks:
            ids=np.flatnonzero(ok & d.comparison_block.eq(b).to_numpy())
            if not len(ids): continue
            n=int(np.ceil(len(ids)*.1)); tie=np.random.RandomState(42).rand(len(ids))
            rank=ids[np.lexsort((tie,-prob[ids]))[:n]]
            inspections+=n; caught+=int(y[rank].sum())
        metrics.append(dict(scheme=scheme,model=name,inputs=len(fs),shots=int(ok.sum()),gas=int(yy.sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),AP=average_precision_score(yy,pp),F1=f1_score(yy,pred,zero_division=0),top10_caught=caught,top10_inspected=inspections))
        for i in np.flatnonzero(ok):
            predictions.append(dict(scheme=scheme,model=name,TimeStamp=d.TimeStamp.iloc[i],comparison_block=d.comparison_block.iloc[i],actual=int(y[i]),probability=float(prob[i]),prediction=int(prob[i]>=.5)))
        print(scheme,name,metrics[-1],flush=True)
mt=pd.DataFrame(metrics)
mt.to_csv(tables/'rg3_local_relation_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(predictions).to_csv(tables/'rg3_local_relation_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(foldlog).to_csv(tables/'rg3_local_relation_fold_log.csv',index=False,encoding='utf-8-sig')
reviews=[
'# 팀 코드 검토 및 RG3 시점별 관계 파생변수 실험','',
'## 검토한 버전','',
'- hayeon에서 작업. taein 701cff4, jiwoo cc2847d의 Git 객체를 읽어 검토. 브랜치 코드 복사·병합 없이 진행.',
'- 태인: notebooks/01_link_primary_secondary.ipynb, notebooks/semi_supervised_guidebook.ipynb.',
'- 지우: src/official.py 및 src/load.py, docs/findings.md, notebooks/05~12의 주요 분석 코드. 과거 초안 findings.md와 최신 공식범위는 표본 수가 다르므로 혼용하지 않는다.',
'','## 팀 코드에서 가져온 점과 주의점','',
'- 태인의 원본·공식 데이터 연결 검증은 우리 원단위 파생변수 방향과 부합한다. 현재 RG3 범위는 동일하고 CN7 10/27은 별도 범위.',
'- 태인 가이드북 재현의 기본 SPLIT_MODE는 guidebook(행 단위)이다. shot 옵션은 LH/RH를 같은 분할에 두지만, unlabeled/test 중복 제거와 파일별 표준화 차이 보정은 해당 노트북 코드에 없다. 그대로 성능 비교에 사용하지 않았다.',
'- 지우 official.py의 window=5 짝 묶기는 초기 05 노트북의 인접 행 묶기보다 보완돼 있다. 원본 Timestamp가 있는 우리 쪽은 설비·제품·시각 키를 계속 사용한다.',
'- 지우 07~12는 과거 기준선, 변수 관계 복원, 순위 검사 및 오탐 관리 아이디어가 유용하다.',
'- 지우 08/10/12의 과거 비라벨 기준선은 시각 필터 중심이며 설비별 분리·제품 전환·긴 정지 간격 초기화가 함수 안에 없다. 우리 실험은 설비·제품별 30분 이력 초기화 규칙을 유지한다.',
'- 지우 12의 양품 모드는 배럴1>100 등 공정 상태 필터이지 품질 정상 정답이 아니다. 비라벨을 확정 정상으로 부르지 않는다.',
'- 지우 12 컨포멀은 현재 점수를 비라벨 스트림 Timestamp에서 찾으므로 비라벨에 없는 라벨 시각은 p=1로 돌아간다. 비라벨 겹침 여부에 따라 검출 기회가 달라질 수 있다. 현재 노트북은 모든 라벨 Shot에서 현재 점수를 직접 계산한다.',
'- 가중 과거 순위는 공정 드리프트와 비라벨 품질혼합에서 자동으로 5% 정상 오탐을 보장하는 것은 아니다.',
'- 지우 12 Deep SAD는 다른 블록의 미래 불량도 참고할 수 있다. 구간 외부 평가와 미래 예측 평가를 구분해야 한다.',
'- 센서 동일성 F1 상한은 양쪽 행 동일입력 조건에 한정. RH 전용 현재 작업에는 그대로 적용하지 않는다.',
'','## 이번에 추가한 변수','',
'1. local_pressure_relation_z: 직전 최대30개의 압력~속도 관계로 현재 압력을 예측한다. 앞쪽 이력으로 Ridge를 맞추고 마지막5개 과거 이력의 예측오차로 현재 이탈 정도를 표준화한다. 최소15개 이력 필요.',
'2. local_cushion_z: 현재 쿠션이 직전10개 쿠션의 중앙값·변동폭에서 얼마나 벗어났는지(부호 유지). 최소5개 필요.',
'3. local_barrel2_minus5_z: 현재 배럴2−5 온도차가 직전30개 같은 온도차의 평소 수준에서 얼마나 벗어났는지. 최소5개 필요.',
'- 값이 일정해 표준화 분모를 정할 수 없으면 결측으로 두고 학습 자료 중앙값으로 대체한다. 거짓 0점을 강제로 만들지 않는다.',
'- 새 3변수 조합, 압력·쿠션 이탈+배럴2·5 과거 흔들림의 4변수 조합만 비교. 기본 흔들림4변수는 그대로 재실행.',
'- 변수 수와 원센서 수는 다르다. 온도차 변수는 두 센서를 참고한다. 계산식은 검증할 통계 가설이며 가스 원인식이 아니다.',
'','## 고정 평가 조건','',
'- RH 가스17 + 정상566. 미성형8 제외. LH 합치지 않음.',
'- RF 200트리, 깊이8, leaf3, balanced_subsample, seed42. 확률0.5 고정. 결측 대체는 학습 자료만.',
'- 구간별 전체 제외: 기존과 동일한583개 비교. 미래 구간이 학습에 들어갈 수 있음.',
'- 과거 구간만 학습: 두 클래스 학습이 가능한 후반3구간만 평가. 초반 미평가 구간은 실패0점으로 섞지 않음.',
'- 현재/미래 센서 및 라벨을 과거 관계 학습에 넣지 않음. 단, 시험구간 앞선 Shot 센서를 이력에 쓰는 온라인 관측 가능 조건이다.',
'- 테스트 Timestamp 및 연속 센서 이력이 제공되지 않으면 현재 파생변수를 대회 테스트에서 그대로 만들 수 없다.',
'- 같은 자료를 여러 차례 분석한 뒤 정한 새 가설이므로 과거순 평가도 완전히 손대지 않은 독립 테스트는 아니다.',
'','## 결과','']
def tab(frame):
    cols=['model','inputs','shots','gas','TP','FP','FN','TN','AP','F1','top10_caught','top10_inspected']
    lines=['| '+' | '.join(cols)+' |','|'+'|'.join(['---']*len(cols))+'|']
    for _,r in frame.iterrows():
        lines.append('| '+' | '.join(f'{r[c]:.4f}' if c in ['AP','F1'] else str(r[c]) for c in cols)+' |')
    return lines
for scheme,group in mt.groupby('scheme',sort=False):
    reviews+=['','### '+scheme,'']+tab(group)
reviews+=['','## 변수 이력 확보(행 수)','',coverage.to_string(),'',
'## 판단','',
'새 후보가 TP를 늘려도 FP가 같이 크게 늘면 사용 가능한 개선이라고 하지 않는다. 기존과 같은 대상·같은 분할의 결과만 직접 비교하고, 후반3구간 지표는583개 전체 지표와 섞지 않는다. 후보 선택은 탐색 결과이며 독립 보류 자료 검증이 추가로 필요하다.',
'','코드 출처: [태인 브랜치](https://github.com/hayeon4060/KAMP-IMM/tree/701cff4705a7f6ee12a47e2ac362bdf070f38957), [지우 브랜치](https://github.com/hayeon4060/KAMP-IMM/tree/cc2847d4e617d37e3140e35ebe387f1a179e48b1).']
(root/'reports/rg3_team_code_and_local_relation_review.md').write_text('\n'.join(reviews),encoding='utf-8')
print('coverage',coverage.to_string())


leave_one_block_out baseline_std4 {'scheme': 'leave_one_block_out', 'model': 'baseline_std4', 'inputs': 4, 'shots': 583, 'gas': 17, 'TP': 1, 'FP': 6, 'FN': 16, 'TN': 560, 'AP': 0.13871113959502832, 'F1': 0.08333333333333333, 'top10_caught': 6, 'top10_inspected': 62}


leave_one_block_out local_relation3 {'scheme': 'leave_one_block_out', 'model': 'local_relation3', 'inputs': 3, 'shots': 583, 'gas': 17, 'TP': 0, 'FP': 5, 'FN': 17, 'TN': 561, 'AP': 0.03943034680976784, 'F1': 0.0, 'top10_caught': 1, 'top10_inspected': 62}


leave_one_block_out local_relation_plus_std4 {'scheme': 'leave_one_block_out', 'model': 'local_relation_plus_std4', 'inputs': 4, 'shots': 583, 'gas': 17, 'TP': 0, 'FP': 4, 'FN': 17, 'TN': 562, 'AP': 0.05430209369988689, 'F1': 0.0, 'top10_caught': 4, 'top10_inspected': 62}


past_blocks_only baseline_std4 {'scheme': 'past_blocks_only', 'model': 'baseline_std4', 'inputs': 4, 'shots': 440, 'gas': 13, 'TP': 0, 'FP': 2, 'FN': 13, 'TN': 425, 'AP': 0.1133015426330235, 'F1': 0.0, 'top10_caught': 4, 'top10_inspected': 46}


past_blocks_only local_relation3 {'scheme': 'past_blocks_only', 'model': 'local_relation3', 'inputs': 3, 'shots': 440, 'gas': 13, 'TP': 0, 'FP': 5, 'FN': 13, 'TN': 422, 'AP': 0.03569038918765434, 'F1': 0.0, 'top10_caught': 2, 'top10_inspected': 46}


past_blocks_only local_relation_plus_std4 {'scheme': 'past_blocks_only', 'model': 'local_relation_plus_std4', 'inputs': 4, 'shots': 440, 'gas': 13, 'TP': 0, 'FP': 2, 'FN': 13, 'TN': 425, 'AP': 0.09704031740448218, 'F1': 0.0, 'top10_caught': 3, 'top10_inspected': 46}


coverage             local_pressure_relation_z  local_cushion_z  local_barrel2_minus5_z
PassOrFail                                                                    
0                                 552              562                     562
1                                  16               16                      16
